# What the Question Parser Extracts from a User String: Keywords, Scope, Shape, Decomposition, Clarification

> 📖 **Full article on Towards Data Science:** [What the Question Parser Extracts from a User String: Keywords, Scope, Shape, Decomposition, Clarification](https://towardsdatascience.com/what-the-question-parser-extracts-from-a-user-string-keywords-scope-shape-decomposition-clarification/)
>
> This is the **runnable, code-only companion**. The explanations, the diagrams and the *why* live in the article. Here you just run the pipeline end to end. Every section below links back to the matching part of the article. **To understand any step, read the article.**


In [ ]:
%load_ext autoreload
%autoreload 2

# Bootstrap: add src/ to sys.path so we can import docintel without pip install.
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:
        if (p / ".git").exists():
            return p
        p = p.parent
    return start.resolve()

REPO_ROOT = _find_repo_root(Path.cwd())
for sub in ("",):
    candidate = str(REPO_ROOT / sub)
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

In [ ]:
# Load two public PDFs to a tiny page_df (one row per page, full text).
paper_pdf = REPO_ROOT / "data" / "paper" / "1706.03762v7.pdf"
nist_pdf = REPO_ROOT / "data" / "nist" / "NIST.CSWP.29.pdf"

def load_page_df(pdf_path: Path) -> pd.DataFrame:
    doc = fitz.open(str(pdf_path))
    return pd.DataFrame([
        {"page": i + 1, "text": page.get_text()}
        for i, page in enumerate(doc)
    ])

paper_page_df = load_page_df(paper_pdf)
nist_page_df = load_page_df(nist_pdf)
print(f"Attention paper: {len(paper_page_df)} pages")
print(f"NIST CSF 2.0:    {len(nist_page_df)} pages")

In [ ]:
# Stubs for types referenced in passing in the article (e.g. §2.7's ExecutionPlan
# is grouped with illustrative pseudo-code that calls undefined helpers, so the whole
# block is fenced as illustration; we still need ExecutionPlan defined here for §2.7
# and §4.1 cells to construct instances). Same for DocumentProfile (§4.1 type annotation)
# and Decomposition (§2.4 grouped with illustrative `decompose` stub).
class ExecutionPlan(BaseModel):
    use_toc_navigation: bool = True
    use_keyword_retrieval: bool = True
    use_embeddings: bool = False
    follow_cross_references: bool = False
    decompose_compound: bool = False
    iterate_on_feedback: bool = True
    extract_page_numbers: bool = True

class DocumentProfile(BaseModel):
    format: str = "pdf"
    has_toc: bool = True

class Decomposition(BaseModel):
    pattern: Literal["single", "independent", "sequential", "unified", "conditional"] = "single"
    sub_questions: list[str] = Field(default_factory=list)

## 1. The five field families the parser fills

📖 **Read section 1 of the article on TDS:** [What the Question Parser Extracts from a User String: Keywords, Scope, Shape, Decomposition, Clarification](https://towardsdatascience.com/what-the-question-parser-extracts-from-a-user-string-keywords-scope-shape-decomposition-clarification/)


In [ ]:
demo_q = "How does multi-head atention compare to self-atention?"
corrected = correct_spelling(demo_q)
print(f"raw:       {demo_q}")
print(f"corrected: {corrected}")

In [ ]:
for q, dom in [
    ("What is the dimension of the embeddings?", "deep learning research paper"),
    ("What does the framework recommend for asset management?", "cybersecurity standard"),
]:
    print(f"Q: {q}  (domain: {dom})")
    for r in rewrite_query(q, domain_hint=dom):
        print(f"  - {r}")
    print()

In [ ]:
concepts_df = pd.DataFrame([
    {"document_type": "insurance", "concept": "premium",
     "definition": "The amount the policyholder pays for coverage.",
     "default_chunk_strategy": "sequential", "default_answer_context": "line",
     "default_needs_summary": False, "default_model": None},
    {"document_type": "insurance", "concept": "non_compete",
     "definition": "A clause barring an employee or party from competing.",
     "default_chunk_strategy": "combined",   "default_answer_context": "section",
     "default_needs_summary": True,  "default_model": "gpt-4.1"},
    {"document_type": "insurance", "concept": "exclusions",
     "definition": "Items the contract specifically excludes from coverage.",
     "default_chunk_strategy": "combined",   "default_answer_context": "chapter",
     "default_needs_summary": True,  "default_model": "claude-opus-4.7"},
])

In [ ]:
concept_keywords_df = pd.DataFrame([
    {"concept": "premium",     "language": "fr", "keyword": "prime",                     "keyword_priority": "primary",   "weight": 1.0},
    {"concept": "premium",     "language": "fr", "keyword": "cotisation",                "keyword_priority": "primary",   "weight": 1.0},
    {"concept": "premium",     "language": "fr", "keyword": "tarif annuel",              "keyword_priority": "secondary", "weight": 0.5},
    {"concept": "premium",     "language": "en", "keyword": "premium",                   "keyword_priority": "primary",   "weight": 1.0},
    {"concept": "premium",     "language": "en", "keyword": "annual fee",                "keyword_priority": "secondary", "weight": 0.5},
    {"concept": "non_compete", "language": "en", "keyword": "non-compete",               "keyword_priority": "primary",   "weight": 1.0},
    {"concept": "non_compete", "language": "en", "keyword": "restrictive covenant",      "keyword_priority": "secondary", "weight": 0.5},
    {"concept": "non_compete", "language": "fr", "keyword": "clause de non-concurrence", "keyword_priority": "primary",   "weight": 1.0},
])

# Lookup helper: given a question and the keyword dictionary, pull every variant
# of every concept the question matches. Pandas only, no LLM.
def lookup_expert_keywords(
    question: str,
    ckw_df: pd.DataFrame,
    *,
    language: str | None = None,
) -> pd.DataFrame:
    """Pull every variant of every concept whose keyword appears in `question`.
    `language` filters the dictionary to one language (the question's). Pass
    None to search every language at once on mixed-language corpora."""
    q_lower = question.lower()
    pool = ckw_df if language is None else ckw_df[ckw_df["language"] == language]
    # Word-boundary match avoids 'cap' inside 'capacity', 'prime' inside 'principal'.
    hits = pool[pool["keyword"].apply(
        lambda kw: bool(re.search(rf"\b{re.escape(kw.lower())}\b", q_lower))
    )]
    if hits.empty:
        return ckw_df.iloc[0:0]
    matched_concepts = hits["concept"].unique().tolist()
    return ckw_df[ckw_df["concept"].isin(matched_concepts)].reset_index(drop=True)

In [ ]:
for q in [
    "Does article L131-1 of the insurance code apply here?",
    "What does ID.AM-1 cover in the NIST framework?",
    "What is the value of d_model in the Transformer base?",
]:
    print(f"{str(extract_anchor_keywords(q)):40s}  <-  {q}")

In [ ]:
answer_types_df = pd.DataFrame([
    {"type": "amount",  "retrieval_patterns": [r"\d[\d\s.,]*\s*(?:EUR|€|USD|\$)"],
     "output_schema_ref": "generation.AmountValue",
     "definition": "A monetary value with currency.",
     "default_model": "gpt-4.1-nano"},
    {"type": "date",    "retrieval_patterns": [r"\d{4}-\d{2}-\d{2}", r"\d{1,2}\s+\w+\s+\d{4}"],
     "output_schema_ref": "generation.DateValue",
     "definition": "A calendar date in any common format.",
     "default_model": "gpt-4.1-nano"},
    {"type": "iban",    "retrieval_patterns": [r"\b[A-Z]{2}\d{2}[A-Z0-9]{1,30}\b"],
     "output_schema_ref": "generation.IbanValue",
     "definition": "A bank account number in IBAN format.",
     "default_model": "gpt-4.1-nano"},
    {"type": "address", "retrieval_patterns": [r"\b\d{5}\b"],
     "output_schema_ref": "generation.AddressValue",
     "definition": "A postal address (street, city, zip, country).",
     "default_model": "gpt-4.1-mini"},
    {"type": "text",    "retrieval_patterns": [],
     "output_schema_ref": "generation.TextValue",
     "definition": "Free-form text fallback.",
     "default_model": "gpt-4.1-mini"},
])

In [ ]:
answer_shapes_df = pd.DataFrame([
    {"shape": "single",      "default_chunk_strategy": "sequential", "default_answer_context": "line"},
    {"shape": "listing",     "default_chunk_strategy": "combined",   "default_answer_context": "section"},
    {"shape": "table",       "default_chunk_strategy": "combined",   "default_answer_context": "section"},
    {"shape": "tree",        "default_chunk_strategy": "combined",   "default_answer_context": "section"},
    {"shape": "nested_json", "default_chunk_strategy": "combined",   "default_answer_context": "paragraph"},
])

In [ ]:
for q in [
    "In the warranty section, what's the liability cap?",
    "What does it say on page 3 of the contract?",
    "What are the exclusions of this contract?",
]:
    print(f"{q}\n  -> {extract_hints(q)}\n")

In [ ]:
class Decomposition(BaseModel):
    pattern: Literal['single', 'independent', 'sequential',
                     'unified', 'conditional'] = 'single'
    sub_questions: list[str] = Field(default_factory=list)
    conditional_filter: dict | None = None

def decompose(question: str) -> Decomposition:
    if not has_compound_indicators(question):
        return Decomposition(pattern='single')
    return llm_classify_decomposition(question)

---

📖 **Read the full article on TDS:** [What the Question Parser Extracts from a User String: Keywords, Scope, Shape, Decomposition, Clarification](https://towardsdatascience.com/what-the-question-parser-extracts-from-a-user-string-keywords-scope-shape-decomposition-clarification/)

The whole series ships on Towards Data Science: [all articles by Angela Shi](https://towardsdatascience.com/author/angela-shi/)

This notebook ships a runnable slice. **For the complete code** (every brick, the dispatcher, the schemas), get in touch on Ko-fi: https://ko-fi.com/angelashi
